# 🍜 STEP 06 — 에이전트 간 데이터 교류
## 상권분석 → 재무분석 → 상권 재조회 파이프라인

### 학습 포인트
| 개념 | 설명 |
|------|------|
| Step 간 데이터 전달 | `context.emit_event(data=...)` 로 raw+분석결과 동시 전달 |
| 조건부 분기 | 재무 결과에 따라 상권 재조회 여부 결정 |
| 에이전트 공유 상태 | `KernelProcessStepState` 로 누적 컨텍스트 유지 |

### 데이터 흐름
```
[Step1] LocationAgent (상권분석)
    ↓  sales_raw + store_raw + analysis
[Step2] FinanceAgent (재무분석)
    ↓  bep + payback + scenarios + recommendation
[Step3] RecommendAgent (조건부 상권 재조회)  ← BEP 달성 어려울 때만
    ↓  alternative_locations
[Step4] ReportAgent (최종 종합 리포트)
```


In [1]:
import asyncio, os, json
from enum import Enum
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.functions import KernelArguments, kernel_function
from semantic_kernel.connectors.mcp import MCPSsePlugin

from semantic_kernel.processes import ProcessBuilder
from semantic_kernel.processes.kernel_process import (
    KernelProcessStep,
    KernelProcessStepContext,
    KernelProcessStepState,
)
from semantic_kernel.processes.kernel_process.kernel_process_event import KernelProcessEvent
from semantic_kernel.processes.local_runtime.local_kernel_process import start

load_dotenv()

def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        api_key=os.getenv('AZURE_OPENAI_API_KEY'),
        api_version='2024-08-01-preview',
    ))
    return k

def make_settings(auto=True):
    s = AzureChatPromptExecutionSettings()
    if auto:
        s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

print('임포트 완료')


임포트 완료


## 1. 이벤트 정의

각 Step 완료 시 emit할 이벤트. **data 필드로 다음 Step에 전달할 딕셔너리를 실어 보냅니다.**

In [2]:
class PipelineEvent(str, Enum):
    # 진입
    Start               = "Start"

    # Step1 → Step2
    LocationDone        = "LocationDone"
    # data: {
    #   "request":        str,          # 원본 요청
    #   "location":       str,          # 지역명
    #   "business_type":  str,          # 업종
    #   "quarter":        str,          # 분기코드
    #   "sales_raw":      dict,         # get_estimated_sales 원본
    #   "store_raw":      dict,         # get_store_count 원본
    #   "location_analysis": str,       # LocationAgent 분석 텍스트
    # }

    # Step2 → Step3 (BEP 어려움 → 상권 재조회)
    FinanceDoneNeedAlt  = "FinanceDoneNeedAlt"
    # data: 위 + {
    #   "finance_analysis": str,
    #   "bep_monthly_sales": float,     # 손익분기 월매출 (원)
    #   "payback_months":   int,        # 투자회수 예상 개월
    #   "feasibility":      str,        # "어려움" | "보통" | "양호"
    # }

    # Step2 → Step4 (BEP 달성 가능 → 바로 최종 리포트)
    FinanceDoneOk       = "FinanceDoneOk"
    # data: 위와 동일

    # Step3 → Step4
    AltLocationDone     = "AltLocationDone"
    # data: 위 + {
    #   "alt_locations": list[dict]     # 대안 지역 분석 결과
    # }

print('이벤트 정의 완료')
print('흐름: Start → LocationDone → FinanceDoneNeedAlt/Ok → (AltLocationDone →) 최종')


이벤트 정의 완료
흐름: Start → LocationDone → FinanceDoneNeedAlt/Ok → (AltLocationDone →) 최종


## 2. Step 구현

### Step 1 — 상권분석
MCP 서버에서 raw 데이터를 가져오고, LocationAgent가 분석합니다.
**raw 데이터와 분석 텍스트 모두** 다음 Step으로 전달합니다.

In [3]:
class LocationStep(KernelProcessStep):
    """
    Step1: 상권분석
    - MCP 서버에서 sales_raw, store_raw 수집
    - LocationAgent로 분석 텍스트 생성
    - LocationDone 이벤트로 raw + 분석 모두 전달
    """

    @kernel_function(name='analyze_location')
    async def analyze_location(
        self,
        kernel: Kernel,
        request_data: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        location      = request_data.get('location', '홍대')
        business_type = request_data.get('business_type', '카페')
        quarter       = request_data.get('quarter', '20253')
        original_req  = request_data.get('request', '')

        print(f'\n[Step1] 상권분석 시작: {location} / {business_type} / {quarter}')

        # ── MCP 서버 연결 및 raw 데이터 수집 ──
        mcp_plugin = MCPSsePlugin(
            name='SeoulCommercial',
            description='Seoul commercial area API',
            url='http://localhost:8001/sse',
        )
        await mcp_plugin.connect()
        kernel.add_plugin(mcp_plugin)

        # raw 데이터 직접 수집 (에이전트 호출 전)
        sales_result = await kernel.invoke(
            plugin_name='SeoulCommercial',
            function_name='get_estimated_sales',
            location=location, business_type=business_type, quarter=quarter,
        )
        store_result = await kernel.invoke(
            plugin_name='SeoulCommercial',
            function_name='get_store_count',
            location=location, business_type=business_type, quarter=quarter,
        )

        sales_raw = json.loads(str(sales_result))
        store_raw = json.loads(str(store_result))
        print(f'[Step1] raw 수집 완료: 월매출 {sales_raw.get("monthly_sales_krw",0):,}원 / 점포수 {store_raw.get("store_count",0)}개')

        # ── LocationAgent 분석 텍스트 생성 ──
        agent = ChatCompletionAgent(
            name='LocationAgent',
            instructions=(
                'You are a commercial area analysis expert for F&B startups in Seoul. '
                'You will receive pre-fetched raw data. '
                'Synthesize it into a concise Risk/Opportunity analysis. '
                'At the start write: 📅 데이터 기준: {quarter}. '
                'Keep it under 300 words. Always respond in Korean.'
            ),
            kernel=kernel,
            arguments=KernelArguments(settings=make_settings(auto=False)),
        )

        from semantic_kernel.agents import ChatHistoryAgentThread
        thread = ChatHistoryAgentThread()
        prompt = (
            f'다음 raw 데이터를 분석해주세요.\n'
            f'상권분석 raw: {json.dumps(sales_raw, ensure_ascii=False)}\n'
            f'점포수 raw: {json.dumps(store_raw, ensure_ascii=False)}'
        )
        analysis_text = ''
        async for msg in agent.invoke(messages=prompt, thread=thread):
            analysis_text += str(msg.content)

        print(f'[Step1] 분석 완료 ({len(analysis_text)}자)')

        # ── 다음 Step으로 전달할 데이터 패키징 ──
        payload = {
            'request':           original_req,
            'location':          location,
            'business_type':     business_type,
            'quarter':           quarter,
            'sales_raw':         sales_raw,
            'store_raw':         store_raw,
            'location_analysis': analysis_text,
        }

        await step_context.emit_event(
            process_event=PipelineEvent.LocationDone,
            data=payload,
        )

print('Step1 LocationStep 정의 완료')


Step1 LocationStep 정의 완료


### Step 2 — 재무분석
상권 raw 데이터(월 추정매출)를 기반으로 BEP 계산.
**feasibility 판단에 따라 다른 이벤트를 emit**합니다.

In [4]:
class FinanceStep(KernelProcessStep):
    """
    Step2: 재무분석
    - Step1에서 받은 sales_raw의 monthly_sales_krw 활용
    - 사용자 투자/비용 조건과 결합하여 BEP·투자회수 계산
    - feasibility 판단:
        '어려움' → FinanceDoneNeedAlt (Step3 상권 재조회)
        '보통'/'양호' → FinanceDoneOk (Step4 최종 리포트)
    """

    @kernel_function(name='analyze_finance')
    async def analyze_finance(
        self,
        kernel: Kernel,
        location_payload: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        sales_raw     = location_payload['sales_raw']
        store_raw     = location_payload['store_raw']
        location      = location_payload['location']
        business_type = location_payload['business_type']

        monthly_sales = sales_raw.get('monthly_sales_krw', 0)
        store_count   = store_raw.get('store_count', 0)
        tx_count      = sales_raw.get('monthly_tx_count', 0)

        print(f'\n[Step2] 재무분석 시작')
        print(f'  상권 월매출(전체): {monthly_sales:,}원 / 점포 {store_count}개')
        print(f'  점포당 예상 월매출: {monthly_sales // max(store_count,1):,}원')

        # ── FinanceAgent ──
        agent = ChatCompletionAgent(
            name='FinanceAgent',
            instructions=(
                'You are a financial analyst for F&B startups in Korea. '
                'You receive commercial area data and must calculate: '
                '1) Estimated monthly sales per store (from total area sales / store count) '
                '2) Break-even point (BEP) monthly sales based on typical cost structure '
                '3) Payback period in months '
                '4) Three scenarios: pessimistic(60%), base(100%), optimistic(130%) '
                '5) Feasibility judgment: "어려움"(payback>24mo), "보통"(12-24mo), "양호"(<12mo) '
                'Use Korean won. '
                'Return a JSON block wrapped in ```json ... ``` with keys: '
                'bep_monthly_sales, payback_months, per_store_sales, '
                'scenarios(list of {name,monthly_sales,payback}), feasibility, summary. '
                'Then write a brief Korean explanation after the JSON.'
            ),
            kernel=kernel,
            arguments=KernelArguments(settings=make_settings(auto=False)),
        )

        from semantic_kernel.agents import ChatHistoryAgentThread
        thread = ChatHistoryAgentThread()

        # 상권 raw 데이터를 영어로 전달 (Azure RAI 우회)
        prompt = (
            f'Commercial area data for {location} {business_type}:\n'
            f'- Total area monthly sales: {monthly_sales:,} KRW\n'
            f'- Store count in area: {store_count}\n'
            f'- Monthly transaction count: {tx_count:,}\n'
            f'- Weekday sales: {sales_raw.get("weekday_sales_krw",0):,} KRW\n'
            f'- Weekend sales: {sales_raw.get("weekend_sales_krw",0):,} KRW\n'
            f'- Age 20s sales: {sales_raw.get("age_20s_krw",0):,} KRW\n'
            f'- Age 30s sales: {sales_raw.get("age_30s_krw",0):,} KRW\n'
            f'Assumed investment: 80,000,000 KRW, monthly fixed cost: 4,000,000 KRW, '
            f'variable cost ratio: 35%. Respond in Korean.'
        )

        finance_text = ''
        async for msg in agent.invoke(messages=prompt, thread=thread):
            finance_text += str(msg.content)

        # JSON 블록 파싱
        import re
        finance_data = {}
        m = re.search(r'```json\s*({.*?})\s*```', finance_text, re.DOTALL)
        if m:
            try:
                finance_data = json.loads(m.group(1))
            except:
                pass

        bep           = finance_data.get('bep_monthly_sales', 0)
        payback       = finance_data.get('payback_months', 99)
        feasibility   = finance_data.get('feasibility', '보통')

        print(f'[Step2] BEP: {bep:,}원 / 투자회수: {payback}개월 / 판정: {feasibility}')

        payload = {
            **location_payload,
            'finance_analysis': finance_text,
            'bep_monthly_sales': bep,
            'payback_months':    payback,
            'feasibility':       feasibility,
        }

        # ── 분기: feasibility에 따라 다른 이벤트 emit ──
        if feasibility == '어려움':
            print('[Step2] → 상권 재조회 필요 (FinanceDoneNeedAlt)')
            await step_context.emit_event(
                process_event=PipelineEvent.FinanceDoneNeedAlt,
                data=payload,
            )
        else:
            print('[Step2] → 바로 최종 리포트 (FinanceDoneOk)')
            await step_context.emit_event(
                process_event=PipelineEvent.FinanceDoneOk,
                data=payload,
            )

print('Step2 FinanceStep 정의 완료')


Step2 FinanceStep 정의 완료


### Step 3 — 대안 상권 조회 (조건부)
BEP 달성이 어려울 때만 실행. **재무 결과를 참조**하여 더 나은 인근 지역을 조회합니다.

In [5]:
class AltLocationStep(KernelProcessStep):
    """
    Step3: 대안 상권 조회 (BEP '어려움' 판정 시에만 실행)
    - 재무분석 결과(bep_monthly_sales)를 기준으로
      '이 BEP를 달성할 수 있는 인근 지역'을 MCP로 조회
    - 현재 지역 인근 후보를 순차 조회하여 비교
    """

    # 지역별 인근 후보 매핑
    NEARBY_MAP: dict = {
        '홍대':  ['합정', '연남동', '망원'],
        '강남':  ['역삼', '선릉', '삼성'],
        '이태원': ['한남동', '용산'],
        '건대':  ['성수동', '뚝섬'],
        '잠실':  ['송파', '가락'],
    }

    @kernel_function(name='find_alt_location')
    async def find_alt_location(
        self,
        kernel: Kernel,
        finance_payload: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        location      = finance_payload['location']
        business_type = finance_payload['business_type']
        quarter       = finance_payload['quarter']
        bep           = finance_payload.get('bep_monthly_sales', 0)
        payback       = finance_payload.get('payback_months', 99)

        print(f'\n[Step3] 대안 상권 조회 시작')
        print(f'  현재: {location} / BEP: {bep:,}원 / 투자회수: {payback}개월')

        candidates = self.NEARBY_MAP.get(location, ['합정', '연남동'])
        print(f'  조회 후보: {candidates}')

        # MCP 플러그인 재연결 (Step마다 독립 kernel이므로)
        mcp_plugin = MCPSsePlugin(
            name='SeoulCommercialAlt',
            description='Seoul commercial area API',
            url='http://localhost:8001/sse',
        )
        await mcp_plugin.connect()
        kernel.add_plugin(mcp_plugin)

        alt_results = []
        for candidate in candidates:
            try:
                s = await kernel.invoke(
                    plugin_name='SeoulCommercialAlt',
                    function_name='get_estimated_sales',
                    location=candidate, business_type=business_type, quarter=quarter,
                )
                t = await kernel.invoke(
                    plugin_name='SeoulCommercialAlt',
                    function_name='get_store_count',
                    location=candidate, business_type=business_type, quarter=quarter,
                )
                s_data = json.loads(str(s))
                t_data = json.loads(str(t))

                per_store = s_data.get('monthly_sales_krw', 0) // max(t_data.get('store_count', 1), 1)
                alt_results.append({
                    'location':       candidate,
                    'sales_raw':      s_data,
                    'store_raw':      t_data,
                    'per_store_sales': per_store,
                    'store_count':    t_data.get('store_count', 0),
                })
                print(f'  [{candidate}] 점포당 월매출: {per_store:,}원')
            except Exception as e:
                print(f'  [{candidate}] 조회 실패: {e}')

        # BEP 달성 가능한 지역 필터링
        viable = [r for r in alt_results if r['per_store_sales'] >= bep * 0.8]
        print(f'[Step3] 후보 {len(alt_results)}개 중 BEP 80% 이상 달성 가능: {len(viable)}개')

        payload = {
            **finance_payload,
            'alt_locations': alt_results,
            'viable_locations': viable,
        }

        await step_context.emit_event(
            process_event=PipelineEvent.AltLocationDone,
            data=payload,
        )

print('Step3 AltLocationStep 정의 완료')


Step3 AltLocationStep 정의 완료


### Step 4 — 최종 종합 리포트
모든 이전 Step 결과를 받아서 하나의 리포트로 종합합니다.
상권 재조회가 있었다면 대안 지역 비교도 포함합니다.

In [6]:
class ReportStep(KernelProcessStep):
    """
    Step4: 최종 종합 리포트
    - Step1(상권), Step2(재무), Step3(대안, 선택적) 결과를 모두 받음
    - 하나의 창업 타당성 보고서로 종합
    """

    @kernel_function(name='generate_report')
    async def generate_report(
        self,
        kernel: Kernel,
        full_payload: dict,
        step_context: KernelProcessStepContext,
    ) -> None:
        print('\n[Step4] 최종 리포트 생성 시작')

        location         = full_payload['location']
        business_type    = full_payload['business_type']
        quarter          = full_payload['quarter']
        location_analysis = full_payload.get('location_analysis', '')
        finance_analysis  = full_payload.get('finance_analysis', '')
        feasibility       = full_payload.get('feasibility', '보통')
        bep               = full_payload.get('bep_monthly_sales', 0)
        payback           = full_payload.get('payback_months', 0)
        alt_locations     = full_payload.get('alt_locations', [])

        agent = ChatCompletionAgent(
            name='ReportAgent',
            instructions=(
                'You are a startup consultant for F&B businesses in Korea. '
                'Synthesize all provided analysis into a final investment feasibility report. '
                'Structure: '
                '1. 📊 상권 분석 요약 '
                '2. 💰 재무 타당성 (BEP, 투자회수) '
                '3. 🏆 최종 입지 추천 (대안 지역 있으면 비교표 포함) '
                '4. ✅ 종합 의견 및 실행 권고 '
                'Always respond in Korean. Be specific with numbers.'
            ),
            kernel=kernel,
            arguments=KernelArguments(settings=make_settings(auto=False)),
        )

        from semantic_kernel.agents import ChatHistoryAgentThread
        thread = ChatHistoryAgentThread()

        alt_summary = ''
        if alt_locations:
            alt_summary = '\n대안 지역 조회 결과:\n'
            for a in alt_locations:
                alt_summary += (
                    f"- {a['location']}: 점포당 월매출 {a.get('per_store_sales',0):,}원, "
                    f"점포수 {a.get('store_count',0)}개\n"
                )

        prompt = (
            f'창업 타당성 종합 분석 요청\n\n'
            f'=== 기본 정보 ===\n'
            f'지역: {location} / 업종: {business_type} / 분기: {quarter}\n'
            f'BEP 월매출: {bep:,}원 / 투자회수: {payback}개월 / 판정: {feasibility}\n\n'
            f'=== 상권 분석 결과 ===\n{location_analysis}\n\n'
            f'=== 재무 분석 결과 ===\n{finance_analysis[:800]}\n'
            f'{alt_summary}'
            f'\n위 데이터를 종합하여 최종 창업 타당성 보고서를 작성해주세요.'
        )

        report_text = ''
        async for msg in agent.invoke(messages=prompt, thread=thread):
            report_text += str(msg.content)

        print('\n' + '='*60)
        print('📋 최종 창업 타당성 보고서')
        print('='*60)
        print(report_text)
        print('='*60)

        # 전체 파이프라인 결과 저장
        full_payload['final_report'] = report_text
        print('\n✅ 파이프라인 완료!')

print('Step4 ReportStep 정의 완료')


Step4 ReportStep 정의 완료


## 3. Process 조립

Step 간 데이터 흐름을 `on_event → send_event_to` 로 연결합니다.

In [ ]:
def build_pipeline():
    builder = ProcessBuilder(name="LocationFinancePipeline")

    step1 = builder.add_step(LocationStep)
    step2 = builder.add_step(FinanceStep)
    step3 = builder.add_step(AltLocationStep)
    step4 = builder.add_step(ReportStep)

    # ── Start → Step1 ──
    builder.on_input_event(PipelineEvent.Start).send_event_to(
        step1, function_name="analyze_location", parameter_name="request_data"
    )

    # ── Step1 → Step2 ──
    step1.on_event(PipelineEvent.LocationDone).send_event_to(
        step2, function_name="analyze_finance", parameter_name="location_payload"
    )

    # ── Step2 → Step3 (BEP 어려움) ──
    step2.on_event(PipelineEvent.FinanceDoneNeedAlt).send_event_to(
        step3, function_name="find_alt_location", parameter_name="finance_payload"
    )

    # ── Step2 → Step4 (BEP 달성 가능) ──
    step2.on_event(PipelineEvent.FinanceDoneOk).send_event_to(
        step4, function_name="generate_report", parameter_name="full_payload"
    )

    # ── Step3 → Step4 ──
    step3.on_event(PipelineEvent.AltLocationDone).send_event_to(
        step4, function_name="generate_report", parameter_name="full_payload"
    )

    return builder.build()


pipeline = build_pipeline()
print("파이프라인 조립 완료")
print()
print("데이터 흐름:")
print("  Start")
print("  → [Step1 LocationStep]  : 상권 raw + 분석")
print("  → [Step2 FinanceStep]   : BEP, 투자회수, feasibility 판정")
print("  → (어려움) [Step3 AltLocationStep] : 인근 대안 상권 조회")
print("  → [Step4 ReportStep]    : 최종 종합 리포트")

ValidationError: 1 validation error for ProcessEdgeBuilder
target
  Input should be a valid dictionary or instance of ProcessFunctionTargetBuilder [type=model_type, input_value=KernelFunctionMetadata(na...dditional_properties={}), input_type=KernelFunctionMetadata]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type

## 4. 실행

⚠️ **사전 조건**: `seoul_commercial_mcp_server_sse.py` 실행 중이어야 합니다.

In [ ]:
kernel = make_kernel()

# 테스트 요청 1: 홍대 카페 (feasibility 결과에 따라 분기)
await start(
    process=pipeline,
    kernel=kernel,
    initial_event=KernelProcessEvent(
        id=PipelineEvent.Start,
        data={
            'request':       'I want to open a cafe in Hongdae, Seoul.',
            'location':      '홍대',
            'business_type': '카페',
            'quarter':       '20253',
        },
    ),
)


In [ ]:
# 테스트 요청 2: 강남 한식 (다른 지역/업종)
kernel2 = make_kernel()

await start(
    process=pipeline,
    kernel=kernel2,
    initial_event=KernelProcessEvent(
        id=PipelineEvent.Start,
        data={
            'request':       'I want to open a Korean restaurant in Gangnam.',
            'location':      '강남',
            'business_type': '한식',
            'quarter':       '20253',
        },
    ),
)


## 5. 핵심 패턴 정리

### Step 간 데이터 전달
```python
# emit 시 data= 로 딕셔너리 전달
await step_context.emit_event(
    process_event=PipelineEvent.LocationDone,
    data=payload,          # ← 다음 Step의 파라미터로 전달됨
)

# 받는 쪽 Step의 함수 파라미터명 = send_event_to의 parameter_name
step1.on_event(PipelineEvent.LocationDone) \
    .send_event_to(step2.functions['analyze_finance'],
                   parameter_name='location_payload')  # ← 함수 파라미터명과 일치
```

### 조건부 분기
```python
# 같은 Step에서 조건에 따라 다른 이벤트 emit
if feasibility == '어려움':
    await step_context.emit_event(PipelineEvent.FinanceDoneNeedAlt, data=payload)
else:
    await step_context.emit_event(PipelineEvent.FinanceDoneOk, data=payload)

# ProcessBuilder에서 각각 다른 Step으로 연결
step2.on_event(PipelineEvent.FinanceDoneNeedAlt).send_event_to(step3...)
step2.on_event(PipelineEvent.FinanceDoneOk).send_event_to(step4...)
```

### 데이터 누적 패턴
```python
# 각 Step에서 이전 payload를 확장(**언패킹)해서 전달
payload = {
    **location_payload,      # Step1 결과 그대로 유지
    'finance_analysis': ..., # Step2 결과 추가
    'bep_monthly_sales': ...,
}
# → Step4는 Step1+2+3 결과를 모두 한 번에 받음
```
